QN.Given a dataset containing timestamp and user_id columns, write a PySpark program to assign session IDs to user activities. Use the following rules to define session boundaries:

A session expires if there is no activity (inactivity) for 30 minutes or more.

A new session starts automatically every 2 hours, even if activity is continuous.

The output should include the original columns (user_id and timestamp) along with a new column session_id, which uniquely identifies each session.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, lag, when, sum
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import datetime

# Initialize SparkSession
spark = SparkSession.builder.appName("SessionIDGeneration").getOrCreate()

# Sample data
data = [
    ("2021-05-01T11:00:00Z", "u1"),
    ("2021-05-01T13:13:00Z", "u1"),
    ("2021-05-01T15:00:00Z", "u2"),
    ("2021-05-01T11:25:00Z", "u1"),
    ("2021-05-01T15:15:00Z", "u2"),
    ("2021-05-01T02:15:00Z", "u3"),
    ("2021-05-01T02:15:00Z", "u4"),
]

# Define schema
schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("user_id", StringType(), True)
])

# Create DataFrame from hardcoded data
df = spark.createDataFrame(data, schema)

# Convert timestamp column to proper TimestampType
df = df.withColumn("timestamp", col("timestamp").cast(TimestampType()))

# Define a window specification to partition by user_id and order by timestamp
window_spec = Window.partitionBy("user_id").orderBy("timestamp")

# Calculate the time difference in seconds between consecutive events for each user
df_with_lags = df.withColumn(
    "prev_timestamp", lag("timestamp").over(window_spec)
).withColumn(
    "time_diff",
    (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp"))
)

# Define the session boundary conditions
# Condition 1: Inactivity for 30 minutes (1800 seconds)
# Condition 2: A new session starts every 2 hours (7200 seconds)
df_with_sessions = df_with_lags.withColumn(
    "is_new_session",
    when(col("time_diff").isNull(), 1)  # First event is always a new session
    .when(col("time_diff") >= 7200, 1)  # Session starts every 2 hours
    .when(col("time_diff") >= 1800, 1)  # Inactivity threshold of 30 minutes
    .otherwise(0)
)

# Generate session IDs by cumulatively summing "is_new_session"
df_with_sessions = df_with_sessions.withColumn(
    "session_id",
    sum("is_new_session").over(window_spec)
)

# Select the required columns
result_df = df_with_sessions.select("user_id", "timestamp", "session_id")

# Show the resulting DataFrame
result_df.show()


+-------+-------------------+----------+
|user_id|          timestamp|session_id|
+-------+-------------------+----------+
|     u1|2021-05-01 11:00:00|         1|
|     u1|2021-05-01 11:25:00|         1|
|     u1|2021-05-01 13:13:00|         2|
|     u2|2021-05-01 15:00:00|         1|
|     u2|2021-05-01 15:15:00|         1|
|     u3|2021-05-01 02:15:00|         1|
|     u4|2021-05-01 02:15:00|         1|
+-------+-------------------+----------+

